# Build MDE webapp data (`docs/data/mde.json`)

Exports the canonical Replogle-pipeline-modified MDE (coords + leiden/hdbscan clusters + enrichment annotations) plus per-pert n_DEGs as a compact JSON the static viewer (`docs/`) consumes.

**Inputs**
- `KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Filtered_Replogle_Pipeline_Modified_MDE_clusters_annotated.xlsx` — 3 sheets: `MDE` (1,656 perts with x, y, gene_target, leiden, hdbscan), `leiden cluster` (62 rows of CORUM/KEGG/GO/Reactome annotations), `hdbscan cluster` (30 rows of same). Produced by `psp.da.plot_mde` → `psp.da.annotate_clusters` in `KOLF_Perturbation_Atlas_Data_Analysis.ipynb`.
- `psp/notebooks/input_files/n_degs_k562_kolf_rpe1.csv` — per-pert n_DEGs across cell types (we take the `KOLF` column).

**Output**
- `docs/data/mde.json`:
    ```
    {
      "points": [{"g", "x", "y", "l", "h", "n"}, ...],
      "leiden_labels":  {<cluster_id>: <short annotation>},
      "hdbscan_labels": {<cluster_id>: <short annotation>}
    }
    ```

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path('/tscc/projects/ps-malilab/ydoctor/KOLF_Perturbation_Atlas')
XLSX  = ROOT / 'KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Filtered_Replogle_Pipeline_Modified_MDE_clusters_annotated.xlsx'
NDEGS = ROOT / 'psp/notebooks/input_files/n_degs_k562_kolf_rpe1.csv'
OUT   = ROOT / 'docs/data/mde.json'

In [ ]:
# Point coords + cluster ids
mde = pd.read_excel(XLSX, sheet_name='MDE').rename(columns={
    'gene_target': 'gene',
    'leiden cluster': 'leiden',
    'hdbscan cluster': 'hdbscan',
})
mde.head()

In [ ]:
# n_DEGs lookup (CSV is tab-separated, KOLF column holds gene symbols)
ndegs = pd.read_csv(NDEGS, sep='\t', index_col=0)
kolf_ndegs = dict(zip(ndegs['KOLF'].astype(str),
                      pd.to_numeric(ndegs['Number of DEGs_KOLF'], errors='coerce')))
mde['n_degs'] = mde['gene'].map(kolf_ndegs).fillna(-1).astype(int)
print(f'{len(mde)} perts; {int((mde.n_degs < 0).sum())} missing n_DEGs')

In [ ]:
# One short label per cluster, picked from annotation sheets.
# Priority: GO_BP (most general biology) > CORUM (specific complex) > KEGG > Reactome.
# Take the first term before the first ';' and strip any trailing GO id.
ANNOTATION_PRIORITY = (
    'GO_Biological_Process_2025_Annotation',
    'CORUM_Annotation',
    'KEGG_2021_Human_Annotation',
    'Reactome_Pathways_2024_Annotation',
)

def pick_label(row) -> str:
    for col in ANNOTATION_PRIORITY:
        v = str(row.get(col, ''))
        if v and v.strip() and v.lower() != 'nan' and 'no significant' not in v.lower():
            term = v.split(';')[0].strip()
            return re.sub(r'\s*\(GO:\d+\)\s*$', '', term).strip()
    return ''

lei = pd.read_excel(XLSX, sheet_name='leiden cluster')
hdb = pd.read_excel(XLSX, sheet_name='hdbscan cluster')
leiden_labels  = {str(int(r['leiden cluster'])):  pick_label(r) for _, r in lei.iterrows()}
hdbscan_labels = {str(int(r['hdbscan cluster'])): pick_label(r) for _, r in hdb.iterrows()}
print(f"leiden  labeled: {sum(bool(v) for v in leiden_labels.values())}/{len(leiden_labels)}")
print(f"hdbscan labeled: {sum(bool(v) for v in hdbscan_labels.values())}/{len(hdbscan_labels)}")

In [ ]:
points = [
    {'g': r.gene, 'x': round(float(r.x), 3), 'y': round(float(r.y), 3),
     'l': int(r.leiden), 'h': int(r.hdbscan), 'n': int(r.n_degs)}
    for r in mde.itertuples(index=False)
]

payload = {
    'points': points,
    'leiden_labels': leiden_labels,
    'hdbscan_labels': hdbscan_labels,
}

OUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUT, 'w') as f:
    json.dump(payload, f, separators=(',', ':'))

print(f'wrote {OUT} ({OUT.stat().st_size/1024:.1f} KB, {len(points)} points)')

## Preview locally

```bash
cd docs && python -m http.server 8000
# then open http://localhost:8000
```

## Deploy on GitHub Pages

1. Commit `docs/` and push to `main`.
2. GitHub → repo **Settings → Pages**: set **Source = Deploy from a branch**, **Branch = `main`**, **folder `/docs`**. Save.
3. Site is live at `https://y-doctor.github.io/KOLF2.1J_Perturbation_Cell_Atlas/`.